In [12]:
pip install pandas

^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
url = "https://www.redfin.com/city/14913/WA/Redmond"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8"
}
response = requests.get(url, headers=headers)
r = requests.get(url, headers=headers, timeout=15)
r.raise_for_status()


In [ ]:
import requests
import pandas as pd
from io import StringIO
from datetime import datetime

def scrape_open_houses(
    city: str = "Redmond",
    state: str = "WA",
    region_id: str = "14913",   # Redfin's internal city ID
    min_beds: int = None,
    max_price: int = None,
    open_house_date: str = None  # "YYYY-MM-DD", defaults to this weekend
) -> list[dict]:
    """
    Scrape open houses from Redfin for a given city.
    Returns list of dicts with address, timing, and property details.
    """

    # --- 1. Use the CSV endpoint instead of HTML ---
    # Much more reliable — returns structured data directly
    csv_url = (
        f"https://www.redfin.com/stingray/api/gis-csv"
        f"?al=1"
        f"&market={state.lower()}"
        f"&region_id={region_id}"
        f"&region_type=6"
        f"&sf=1,2,3,5,6,7"
        f"&num_homes=350"
        f"&uipt=1"                # single family homes
        f"&open_house=true"       # open houses only
    )

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": f"https://www.redfin.com/city/{region_id}/{state}/{city}",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
    }

    # --- 2. Single request with proper error handling ---
    try:
        r = requests.get(csv_url, headers=headers, timeout=15)
        r.raise_for_status()
    except requests.exceptions.HTTPError as e:
        print(f"HTTP error: {e.response.status_code} — Redfin may be blocking the request")
        return []
    except requests.exceptions.Timeout:
        print("Request timed out — try again")
        return []
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return []

    # --- 3. Parse CSV into DataFrame ---
    try:
        df = pd.read_csv(StringIO(r.text))
    except Exception as e:
        print(f"Failed to parse CSV: {e}")
        print("Raw response:", r.text[:500])  # print first 500 chars to debug
        return []

    # --- 4. Check what columns you actually got ---
    print("Columns received:", df.columns.tolist())  # keep this during dev
    print(f"Total listings: {len(df)}")

    # --- 5. Apply filters ---
    if min_beds and "Beds" in df.columns:
        df = df[df["Beds"] >= min_beds]

    if max_price and "Price" in df.columns:
        # Price column often has $ and commas — clean it first
        df["Price"] = df["Price"].replace(r'[\$,]', '', regex=True).astype(float)
        df = df[df["Price"] <= max_price]

    # --- 6. Filter by open house date if provided ---
    if open_house_date and "Open House Time" in df.columns:
        df = df[df["Open House Time"].str.contains(open_house_date, na=False)]

    # --- 7. Return clean structured list ---
    results = []
    for _, row in df.iterrows():
        results.append({
            "address": row.get("Address", ""),
            "city": row.get("City", city),
            "state": row.get("State", state),
            "zip": row.get("Zip", ""),
            "price": row.get("Price", None),
            "beds": row.get("Beds", None),
            "baths": row.get("Baths", None),
            "sqft": row.get("Sq Ft", None),
            "open_house_time": row.get("Open House Time", None),  # raw string from Redfin
            "url": row.get("URL (SEE https://www.redfin.com/buy-a-home/comparative-market-analysis FOR INFO ON PRICING)", ""),
            "latitude": row.get("Latitude", None),
            "longitude": row.get("Longitude", None),
        })

    return results


# --- Test it ---
if __name__ == "__main__":
    houses = scrape_open_houses(
        city="Redmond",
        state="WA",
        region_id="14913",
        min_beds=3,
        max_price=1500000
    )
    print(f"\nFound {len(houses)} open houses")
    for h in houses[:3]:  # print first 3
        print(h)

In [11]:
soup

<!DOCTYPE html>
<html lang="en"><head>
<script charset="UTF-8" data-domain-script="7e5bc3d6-ef20-4760-aa0d-c8df4649fae2" src="https://cdn.cookielaw.org/scripttemplates/otSDKStub.js" type="text/javascript"></script>
<script>
			window.__uspapi = function (command, version, callback) {
				callback({}, false);
			}
		</script>
<!-- Server: customer-pages-map-dp --><!-- Time generated: Tue May 12 2026 05:05:09 GMT+0000 (Coordinated Universal Time) --><script>(function(a){window.__reactServerOnClickHandler=function(i){(a[i]=a[i]||[]).push(window.event)}})(window.__reactServerUnhandledEvents={})</script><script>
/*! LAB.js (LABjs :: Loading And Blocking JavaScript)
    v2.0.3 (c) Kyle Simpson
    MIT License
*/
!function(t){function e(e){if(t.fetch){var n="The following resources did not resolve within "+y+" ms: "+e,r=JSON.stringify({count:1,errors:[[n]]});t.fetch("/corv/beacon/error",{method:"post",body:"b-"+r}),h&&d(n)}}function n(t,e){p.push([t,+(e||0),+new Date])}function r(t){var e=/^\

In [ ]:
def fetch_and_process(url):
    """Fetch page, extract text + links, and return hash + signals"""
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers, timeout=15)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    text = soup.get_text(separator=" ").lower()
    links = [a.get("href", "") for a in soup.find_all("a")]